# 第 10 章: ロジスティック回帰とアンサンブル学習の探索と可視化

ロジスティック回帰の損失の推移と重み、モデル別の特徴量の重要度、森の大きさと正解率を確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)
@file:DependsOn("org.tribuo:tribuo-classification-tree:4.3.2")
@file:DependsOn("org.tribuo:tribuo-classification-sgd:4.3.2")

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter02.prepareIris
import chapter03.DecisionTree
import chapter10.LogisticRegression
import chapter10.RandomForest
import chapter10.evaluate
import chapter10.forestImportances
import chapter10.models
import chapter10.treeImportances
import chapter10.tribuoRandomForest
import java.io.File
import java.util.logging.Level
import java.util.logging.Logger

// Tribuo が学習の経過を標準エラーに出すので、警告以上だけにする
Logger.getLogger("org.tribuo").level = Level.WARNING

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val irisCsv = File(dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }, "iris.csv")
val split = prepareIris(irisCsv, testSize = 0.3, seed = 0)

## モデルごとの正解率

In [ ]:
val scores = models().map { (name, model) -> name to evaluate(model, split) }
dataFrameOf(
    "モデル" to scores.map { it.first },
    "訓練データ" to scores.map { it.second.train },
    "テストデータ" to scores.map { it.second.test },
)

## ロジスティック回帰の損失の推移

In [ ]:
val logistic = LogisticRegression().fit(split.xTrain, split.tTrain)
val losses =
    dataFrameOf(
        "繰り返し回数" to logistic.losses.indices.map { it + 1 },
        "交差エントロピー" to logistic.losses,
    )
losses.plot {
    line {
        x("繰り返し回数")
        y("交差エントロピー")
    }
    layout.title = "勾配降下法の繰り返し回数と損失"
}

In [ ]:
val checkpoints = listOf(1, 10, 100, 1000, 5000)
dataFrameOf(
    "繰り返し回数" to checkpoints,
    "交差エントロピー" to checkpoints.map { logistic.losses[it - 1] },
)

In [ ]:
val weightColumns =
    listOf("特徴量" to split.xTrain.columnNames()) +
        logistic.classes.mapIndexed { k, label -> label to logistic.weights.map { it[k] } }
dataFrameOf(*weightColumns.toTypedArray())

## モデル別の特徴量の重要度

In [ ]:
val tree = checkNotNull(DecisionTree(3).fit(split.xTrain, split.tTrain).tree)
val forest = RandomForest(nEstimators = 100, maxFeatures = 2, seed = 0).fit(split.xTrain, split.tTrain)
val importances =
    mapOf(
        "決定木（深さ 3）" to treeImportances(tree, split.xTrain, split.tTrain),
        "ランダムフォレスト（100 本）" to forestImportances(forest, split.xTrain, split.tTrain),
    )
val importanceTable =
    dataFrameOf(
        "モデル" to importances.flatMap { (model, values) -> values.keys.map { model } },
        "特徴量" to importances.flatMap { (_, values) -> values.keys },
        "重要度" to importances.flatMap { (_, values) -> values.values },
    )
importanceTable.plot {
    bars {
        x("特徴量")
        y("重要度")
        fillColor("モデル")
    }
    layout.title = "モデル別の特徴量の重要度"
}

In [ ]:
importanceTable

## 森の大きさと正解率

In [ ]:
val sizes = listOf(1, 5, 10, 25, 50, 100)
val sizeScores =
    sizes.map { n ->
        Triple(
            n,
            evaluate(RandomForest(nEstimators = n, maxFeatures = 2, seed = 0), split),
            evaluate(tribuoRandomForest(nEstimators = n, maxDepth = null, seed = 0L), split),
        )
    }
val sizeTable =
    dataFrameOf(
        "木の数" to sizeScores.map { it.first },
        "自作（訓練データ）" to sizeScores.map { it.second.train },
        "自作（テストデータ）" to sizeScores.map { it.second.test },
        "Tribuo（テストデータ）" to sizeScores.map { it.third.test },
    )
sizeTable

In [ ]:
val sizeLong =
    dataFrameOf(
        "木の数" to sizes + sizes + sizes,
        "系列" to sizes.map { "自作（訓練データ）" } + sizes.map { "自作（テストデータ）" } + sizes.map { "Tribuo（テストデータ）" },
        "正解率" to sizeScores.map { it.second.train } + sizeScores.map { it.second.test } + sizeScores.map { it.third.test },
    )
sizeLong.plot {
    line {
        x("木の数")
        y("正解率")
        color("系列")
    }
    points {
        x("木の数")
        y("正解率")
        color("系列")
    }
    layout.title = "ランダムフォレストの木の数と正解率"
}